# Having fun with the IRIS data set
This is a "Hello, World!" project that demonstrates building a neural network to predict the class of flowers based on their features: Petal length/width & Sepal length/width

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import TensorDataset, DataLoader

# -----------------------------
# 1. Load and prepare the data
# -----------------------------
iris = load_iris()
X = iris.data   # shape (150, 4)
y = iris.target # shape (150,)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

# Create DataLoader
train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=16)

# -----------------------------
# 2. Define the neural network
# -----------------------------
class IrisNet(nn.Module):
    def __init__(self):
        super(IrisNet, self).__init__()
        self.fc1 = nn.Linear(4, 5)   # input → hidden1
        # self.fc2 = nn.Linear(5, 5)   # hidden1 → hidden2
        self.fc3 = nn.Linear(5, 3)   # hidden2 → output (3 classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        # x = self.relu(self.fc2(x))
        x = self.fc3(x)  # no softmax, handled by CrossEntropyLoss
        return x

model = IrisNet()

# -----------------------------
# 3. Training setup
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

# -----------------------------
# 4. Training loop
# -----------------------------
epochs = 100
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Evaluate on test set
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            outputs = model(xb)
            _, predicted = torch.max(outputs, 1)
            total += yb.size(0)
            correct += (predicted == yb).sum().item()
    acc = correct / total
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}, Test Acc: {acc:.2f}")

Epoch 1/100, Loss: 1.2024, Test Acc: 0.10
Epoch 2/100, Loss: 1.1458, Test Acc: 0.17
Epoch 3/100, Loss: 1.0857, Test Acc: 0.27
Epoch 4/100, Loss: 1.0286, Test Acc: 0.37
Epoch 5/100, Loss: 0.9881, Test Acc: 0.57
Epoch 6/100, Loss: 0.9210, Test Acc: 0.67
Epoch 7/100, Loss: 0.8231, Test Acc: 0.73
Epoch 8/100, Loss: 0.7544, Test Acc: 0.80
Epoch 9/100, Loss: 0.6714, Test Acc: 0.80
Epoch 10/100, Loss: 0.5889, Test Acc: 0.80
Epoch 11/100, Loss: 0.5385, Test Acc: 0.80
Epoch 12/100, Loss: 0.4838, Test Acc: 0.80
Epoch 13/100, Loss: 0.4421, Test Acc: 0.80
Epoch 14/100, Loss: 0.4314, Test Acc: 0.83
Epoch 15/100, Loss: 0.3936, Test Acc: 0.83
Epoch 16/100, Loss: 0.3825, Test Acc: 0.80
Epoch 17/100, Loss: 0.3525, Test Acc: 0.80
Epoch 18/100, Loss: 0.3297, Test Acc: 0.83
Epoch 19/100, Loss: 0.3150, Test Acc: 0.83
Epoch 20/100, Loss: 0.3107, Test Acc: 0.83
Epoch 21/100, Loss: 0.2914, Test Acc: 0.87
Epoch 22/100, Loss: 0.2712, Test Acc: 0.87
Epoch 23/100, Loss: 0.2700, Test Acc: 0.87
Epoch 24/100, Loss: 

In [2]:
# -----------------------------
# 5. Evaluation on test set
# -----------------------------
model.eval()
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)
    y_pred = predicted.numpy()

# -----------------------------
# 6. Metrics
# -----------------------------
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9666666666666667

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30


Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]
